In [ ]:
import re
import time
import datetime
import logging
import argparse
from pathlib import Path
from dataclasses import dataclass

import requests
from bs4 import BeautifulSoup
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ── logging ───────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-7s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

# ── HTTP session ──────────────────────────────────────────────────────────────
SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": (
        "TrylonCinemaScraper/1.0 "
        "(historical-research; be polite, add delay between requests)"
    ),
    "Accept-Language": "en-US,en;q=0.9",
})
DELAY = 1.2   # seconds between requests

TRYLON_BASE    = "https://trylon.org"
PERISPHERE_BASE = "https://perisphere.org"

# ─────────────────────────────────────────────────────────────────────────────
# HTTP HELPERS
# ─────────────────────────────────────────────────────────────────────────────

def get(url: str, **kwargs) -> requests.Response:
    """GET with 3 retries, exponential back-off, and polite delay."""
    for attempt in range(3):
        try:
            r = SESSION.get(url, timeout=20, **kwargs)
            r.raise_for_status()
            time.sleep(DELAY)
            return r
        except requests.RequestException as exc:
            log.warning("Attempt %d failed for %s : %s", attempt + 1, url, exc)
            time.sleep(DELAY * (attempt + 2))
    raise RuntimeError(f"Failed to fetch {url} after 3 attempts")


def soup(url: str, **kwargs) -> BeautifulSoup:
    return BeautifulSoup(get(url, **kwargs).text, "html.parser")


# ─────────────────────────────────────────────────────────────────────────────
# DATA MODEL
# ─────────────────────────────────────────────────────────────────────────────

@dataclass
class TrylonScreening:
    date:       str   # YYYY-MM-DD  (or "????-MM-DD" when year missing)
    day:        str   # Mon / Tue / …
    showtime:   str   # "7:00 PM"
    film_title: str
    film_year:  str   # e.g. "1985"
    format_:    str   # 35mm / 16mm / DCP / HD
    series:     str   # name of the screening series
    venue:      str   # "Trylon Cinema" / "Heights Theater" / …
    confidence: str   # "Confirmed" / "Partial"
    source_url: str
    notes:      str = ""


# ─────────────────────────────────────────────────────────────────────────────
# DATE / TIME PARSING UTILITIES
# ─────────────────────────────────────────────────────────────────────────────

_MONTH_MAP: dict[str, int] = {
    m: i for i, m in enumerate(
        ["january","february","march","april","may","june",
         "july","august","september","october","november","december"], 1
    )
}
# also accept 3-letter abbreviations
_MONTH_MAP.update({k[:3]: v for k, v in list(_MONTH_MAP.items())})

_DOW = {0:"Mon", 1:"Tue", 2:"Wed", 3:"Thu", 4:"Fri", 5:"Sat", 6:"Sun"}

# Regex: optional day-of-week, then "Month D[D][, YYYY]"
_DATE_RE = re.compile(
    r"(?:(?:Monday|Tuesday|Wednesday|Thursday|Friday|Saturday|Sunday),?\s+)?"
    r"(\w+)\s+(\d{1,2})(?:,?\s*(\d{4}))?",
    re.IGNORECASE,
)
_TIME_RE = re.compile(
    r"\b(\d{1,2}:\d{2}\s*(?:am|pm|AM|PM)|\d{1,2}\s*(?:am|pm|AM|PM)|noon|midnight)\b",
    re.IGNORECASE,
)


def parse_date(raw: str, context_year: int | None = None) -> tuple[str, str]:
    """
    Convert a raw date string to (YYYY-MM-DD, day-of-week).
    Falls back to ("????-MM-DD", "") when the year is missing.
    """
    raw = raw.strip()
    m = _DATE_RE.search(raw)
    if not m:
        return "", ""
    mon_s, day_s, yr_s = m.group(1).lower(), m.group(2), m.group(3)
    mo = _MONTH_MAP.get(mon_s) or _MONTH_MAP.get(mon_s[:3])
    if not mo:
        return "", ""
    year = int(yr_s) if yr_s else context_year
    if year:
        try:
            d = datetime.date(year, mo, int(day_s))
            return d.isoformat(), _DOW[d.weekday()]
        except ValueError:
            pass
    # year unknown
    return f"????-{mo:02d}-{int(day_s):02d}", ""


def parse_time(raw: str) -> str:
    """Extract the first time expression from a string."""
    m = _TIME_RE.search(raw)
    if not m:
        return ""
    t = m.group(1).strip()
    # normalise: "7pm" → "7:00 PM", "noon" → "12:00 PM"
    t_upper = t.upper().replace(".", "")
    if t_upper == "NOON":
        return "12:00 PM"
    if t_upper == "MIDNIGHT":
        return "12:00 AM"
    if ":" not in t:
        hr = re.search(r"\d+", t).group()
        suffix = "PM" if re.search(r"pm", t, re.I) else "AM"
        return f"{int(hr)}:00 {suffix}"
    # ensure uppercase suffix
    return re.sub(r"(am|pm)", lambda x: x.group().upper(), t)


# ─────────────────────────────────────────────────────────────────────────────
# SOURCE 1 — SERIES PAGES  (trylon.org/film-series/)
# ─────────────────────────────────────────────────────────────────────────────

def get_series_urls() -> list[str]:
    """Return every series page URL linked from the series index."""
    log.info("[Series] Fetching index: %s/film-series/", TRYLON_BASE)
    index = soup(f"{TRYLON_BASE}/film-series/")
    seen: set[str] = set()
    urls: list[str] = []
    for a in index.select("a[href]"):
        href = a["href"]
        if "/film-series/" in href and href.rstrip("/") != "/film-series":
            full = href if href.startswith("http") else TRYLON_BASE + href
            if full not in seen:
                seen.add(full)
                urls.append(full)
    log.info("[Series] Found %d series links", len(urls))
    return urls


def scrape_series_page(url: str) -> list[TrylonScreening]:
    """
    Parse one series page.  Each film block looks like:

        <h3>The Terminator (1984)</h3>
        <p>Friday, January 6 at 7:00 PM / Saturday, January 7 at 9:15 PM</p>
        <p>[format info, venue info]</p>

    Date-time pattern scanned: "DayName, Month D[D] at H:MM PM"
    """
    log.info("[Series] Scraping: %s", url)
    try:
        page = soup(url)
    except RuntimeError as exc:
        log.warning("  Skipping: %s", exc)
        return []

    series_name = (page.find("h1") or page.find("h2") or BeautifulSoup("", "html.parser")).get_text(strip=True)
    screenings: list[TrylonScreening] = []

    # Walk every heading that could be a film title
    for heading in page.find_all(["h2", "h3", "h4"]):
        raw_title = heading.get_text(strip=True)
        if not raw_title or len(raw_title) < 3:
            continue
        yr_m = re.search(r"\((\d{4})\)", raw_title)
        film_year  = yr_m.group(1) if yr_m else ""
        film_title = re.sub(r"\s*\(\d{4}\)\s*", "", raw_title).strip()

        # Collect text from following siblings until the next heading
        blob_parts: list[str] = []
        node = heading.find_next_sibling()
        while node and node.name not in ("h2", "h3", "h4"):
            blob_parts.append(node.get_text(" ", strip=True))
            node = node.find_next_sibling()
        blob = " ".join(blob_parts)

        # Detect format and venue
        format_ = "35mm" if "35mm" in blob.lower() else ("16mm" if "16mm" in blob.lower() else "DCP")
        if "heights" in blob.lower():
            venue = "Heights Theater / Trylon Cinema"
        elif "st. anthony" in blob.lower():
            venue = "St. Anthony Main Theatre"
        else:
            venue = "Trylon Cinema"

        # Extract all date+time hits from the blob
        # Pattern: optional DOW, month, day, optional year, optional "at TIME"
        dt_hits = re.findall(
            r"(?:Monday|Tuesday|Wednesday|Thursday|Friday|Saturday|Sunday),?\s+"
            r"(\w+\s+\d{1,2}(?:,?\s*\d{4})?)"
            r"(?:\s+at\s+(\d{1,2}:\d{2}\s*(?:am|pm)))?",
            blob, re.IGNORECASE,
        )

        if dt_hits:
            for date_raw, time_raw in dt_hits:
                date_str, dow = parse_date(date_raw)
                showtime = parse_time(time_raw) if time_raw else "7:00 PM"
                screenings.append(TrylonScreening(
                    date=date_str, day=dow, showtime=showtime,
                    film_title=film_title, film_year=film_year,
                    format_=format_, series=series_name, venue=venue,
                    confidence="Confirmed" if date_str and "????" not in date_str else "Partial",
                    source_url=url,
                ))
        else:
            # Film mentioned on series page but no dates found
            screenings.append(TrylonScreening(
                date="", day="", showtime="",
                film_title=film_title, film_year=film_year,
                format_=format_, series=series_name, venue=venue,
                confidence="Partial", source_url=url,
                notes="Film listed on series page; dates not found",
            ))

    log.info("  → %d screenings", len(screenings))
    return screenings


# ─────────────────────────────────────────────────────────────────────────────
# SOURCE 2 — CALENDAR PAGES  (trylon.org/calendar/?month=YYYY-MM)
# ─────────────────────────────────────────────────────────────────────────────

def scrape_calendar_month(year: int, month: int) -> list[TrylonScreening]:
    """
    Fetch one month from the Trylon event calendar.
    Trylon uses a WordPress events plugin; event cards typically contain:
      - a date attribute on the day cell
      - a link to the individual event page with the film title
      - time in the event excerpt
    """
    url = f"{TRYLON_BASE}/calendar/?month={year}-{month:02d}"
    log.info("[Calendar] %d-%02d  %s", year, month, url)
    try:
        page = soup(url)
    except RuntimeError as exc:
        log.warning("  Skipping: %s", exc)
        return []

    screenings: list[TrylonScreening] = []

    # Try WP event plugin selectors first, then generic article fallback
    events = (
        page.select(".tribe-events-calendar__calendar-event")
        or page.select("article.type-tribe_events")
        or page.select(".tribe_events_cat")
        or page.select("article.event")
    )

    for ev in events:
        # --- date ---
        date_str = ""
        dow = ""
        # Many WP event plugins expose a machine-readable date
        for attr_name in ("datetime", "data-js-month-grid-cell-date", "data-start-date"):
            val = ev.get(attr_name, "")
            if re.match(r"\d{4}-\d{2}-\d{2}", val):
                date_str = val[:10]
                break
        if not date_str:
            time_el = ev.find("time")
            if time_el:
                val = time_el.get("datetime", "")
                m = re.search(r"(\d{4}-\d{2}-\d{2})", val)
                if m:
                    date_str = m.group(1)
        if date_str:
            try:
                d = datetime.date.fromisoformat(date_str)
                dow = _DOW[d.weekday()]
            except ValueError:
                pass

        # --- title ---
        title_el = (
            ev.select_one(".tribe-events-calendar__calendar-event-title a")
            or ev.select_one("h2 a, h3 a, .entry-title a, a")
        )
        film_title = title_el.get_text(strip=True) if title_el else ev.get_text(" ", strip=True)[:60]
        yr_m = re.search(r"\((\d{4})\)", film_title)
        film_year  = yr_m.group(1) if yr_m else ""
        film_title = re.sub(r"\s*\(\d{4}\)\s*", "", film_title).strip()

        if not film_title or len(film_title) < 3:
            continue

        # --- time ---
        blob = ev.get_text(" ", strip=True)
        showtime = parse_time(blob) or "7:00 PM"

        # --- format ---
        format_ = "35mm" if "35mm" in blob.lower() else ("16mm" if "16mm" in blob.lower() else "DCP")

        screenings.append(TrylonScreening(
            date=date_str, day=dow, showtime=showtime,
            film_title=film_title, film_year=film_year,
            format_=format_, series="[Calendar]", venue="Trylon Cinema",
            confidence="Confirmed" if date_str else "Partial",
            source_url=url,
        ))

    log.info("  → %d events", len(screenings))
    return screenings


def scrape_all_calendar_months(start_year: int, end_year: int) -> list[TrylonScreening]:
    all_s: list[TrylonScreening] = []
    for y in range(start_year, end_year + 1):
        for mo in range(1, 13):
            # Don't fetch future months
            if datetime.date(y, mo, 1) > datetime.date.today().replace(day=1):
                break
            all_s.extend(scrape_calendar_month(y, mo))
    return all_s


# ─────────────────────────────────────────────────────────────────────────────
# SOURCE 3 — PERISPHERE BLOG  (perisphere.org)
# ─────────────────────────────────────────────────────────────────────────────

def get_perisphere_post_urls(max_pages: int = 40) -> list[str]:
    """Paginate the Perisphere blog and collect all post URLs."""
    log.info("[Perisphere] Collecting post URLs (max %d index pages) …", max_pages)
    urls: list[str] = []
    seen: set[str] = set()

    for page_num in range(1, max_pages + 1):
        index_url = PERISPHERE_BASE if page_num == 1 else f"{PERISPHERE_BASE}/page/{page_num}/"
        try:
            page = soup(index_url)
        except RuntimeError:
            log.info("  Stopped at index page %d (no more pages)", page_num)
            break

        new_count = 0
        for a in page.select("article a[href], h2.entry-title a, h3.entry-title a, .post-title a"):
            href = a.get("href", "")
            if href and PERISPHERE_BASE in href and href not in seen:
                seen.add(href)
                urls.append(href)
                new_count += 1

        if new_count == 0:
            log.info("  No new posts on index page %d — stopping", page_num)
            break

    log.info("[Perisphere] Found %d post URLs", len(urls))
    return urls


def scrape_perisphere_post(url: str) -> list[TrylonScreening]:
    """
    Parse one Perisphere blog post.

    Posts list upcoming screenings, often in this format:
        Film Title (Year)
        Day, Month D at H:MM pm / Day, Month D at H:MM pm

    We walk line-by-line looking for title lines followed by date lines,
    or lines that contain both a recognisable film title and dates.
    """
    try:
        page = soup(url)
    except RuntimeError as exc:
        log.debug("  Skipping %s : %s", url, exc)
        return []

    # Publish date
    pub_date = ""
    time_el = page.find("time")
    if time_el:
        raw = time_el.get("datetime", "") or time_el.get_text(strip=True)
        m = re.search(r"(\d{4}-\d{2}-\d{2})", raw)
        if m:
            pub_date = m.group(1)
    # fall back to URL slug
    if not pub_date:
        m = re.search(r"(\d{4})[/-](\d{2})[/-](\d{2})", url)
        if m:
            pub_date = f"{m.group(1)}-{m.group(2)}-{m.group(3)}"

    content = page.select_one(".entry-content, .post-content, article, main")
    if not content:
        return []

    # Try to derive context year from post date
    context_year: int | None = None
    if pub_date:
        try:
            context_year = int(pub_date[:4])
        except ValueError:
            pass

    screenings: list[TrylonScreening] = []
    lines = content.get_text("\n", strip=True).split("\n")

    # Walk pairs of lines: title line + date-list line
    i = 0
    while i < len(lines):
        line = lines[i].strip()
        if not line:
            i += 1
            continue

        # A title line: has a film year in parentheses and no date keywords
        yr_m = re.search(r"\((\d{4})\)", line)
        if yr_m and not re.search(
            r"(Monday|Tuesday|Wednesday|Thursday|Friday|Saturday|Sunday)", line, re.I
        ):
            film_year  = yr_m.group(1)
            film_title = re.sub(r"\s*\(\d{4}\)\s*", "", line).strip()
            film_title = re.sub(r"\s*[|—\-].*$", "", film_title).strip()

            # Look at the next 1-3 lines for dates
            for offset in range(1, 4):
                if i + offset >= len(lines):
                    break
                date_line = lines[i + offset].strip()
                dt_hits = re.findall(
                    r"(?:Monday|Tuesday|Wednesday|Thursday|Friday|Saturday|Sunday),?\s+"
                    r"(\w+\s+\d{1,2}(?:,?\s*\d{4})?)"
                    r"(?:\s+at\s+(\d{1,2}:\d{2}\s*(?:am|pm)))?",
                    date_line, re.IGNORECASE,
                )
                if dt_hits:
                    for date_raw, time_raw in dt_hits:
                        date_str, dow = parse_date(date_raw, context_year)
                        showtime = parse_time(time_raw) if time_raw else (
                            parse_time(date_line) or "7:00 PM"
                        )
                        format_ = (
                            "35mm" if "35mm" in date_line.lower()
                            else ("16mm" if "16mm" in date_line.lower() else "DCP")
                        )
                        screenings.append(TrylonScreening(
                            date=date_str, day=dow, showtime=showtime,
                            film_title=film_title, film_year=film_year,
                            format_=format_, series="[Perisphere]", venue="Trylon Cinema",
                            confidence="Confirmed" if date_str and "????" not in date_str else "Partial",
                            source_url=url,
                        ))
        i += 1

    log.debug("[Perisphere] %s → %d screenings", url, len(screenings))
    return screenings


# ─────────────────────────────────────────────────────────────────────────────
# DEDUPLICATION + SORT
# ─────────────────────────────────────────────────────────────────────────────

def _sort_key(s: TrylonScreening) -> datetime.date:
    try:
        return datetime.date.fromisoformat(s.date)
    except ValueError:
        return datetime.date(2099, 1, 1)


def dedupe_and_sort(screenings: list[TrylonScreening]) -> list[TrylonScreening]:
    seen: set[tuple] = set()
    unique: list[TrylonScreening] = []
    for s in screenings:
        key = (s.date, s.film_title[:35].lower().strip(), s.showtime)
        if key not in seen:
            seen.add(key)
            unique.append(s)
    unique.sort(key=_sort_key)
    return unique


# ─────────────────────────────────────────────────────────────────────────────
# EXCEL OUTPUT
# ─────────────────────────────────────────────────────────────────────────────

_DARK_BLUE = "1F3864"
_MED_BLUE  = "2E75B6"
_CONF_COLORS = {
    "Confirmed": "C6EFCE",
    "Partial":   "FFF2CC",
}
_thin = Side(style="thin", color="BFBFBF")
_BD   = Border(left=_thin, right=_thin, top=_thin, bottom=_thin)


def _hdr(cell, txt: str):
    cell.value     = txt
    cell.font      = Font(bold=True, color="FFFFFF", size=9, name="Arial")
    cell.fill      = PatternFill("solid", start_color=_MED_BLUE)
    cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    cell.border    = _BD


def _data(cell, val, bg: str = "DAEEF3", wrap: bool = False, center: bool = False):
    cell.value     = str(val) if val is not None else ""
    cell.font      = Font(size=8, name="Arial")
    cell.fill      = PatternFill("solid", start_color=bg)
    cell.alignment = Alignment(vertical="center", wrap_text=wrap,
                               horizontal="center" if center else "left")
    cell.border    = _BD


def write_xlsx(screenings: list[TrylonScreening], path: str):
    wb = Workbook()
    ws = wb.active
    ws.title = "Trylon Screenings"

    # ── title row ──────────────────────────────────────────────────────────
    ws.merge_cells("A1:J1")
    t = ws["A1"]
    t.value     = ("Trylon Cinema — Historical Screenings "
                   "| 2820 E 33rd St, Minneapolis MN 55406 "
                   "| trylon.org / perisphere.org")
    t.font      = Font(bold=True, size=11, color="FFFFFF", name="Arial")
    t.fill      = PatternFill("solid", start_color=_DARK_BLUE)
    t.alignment = Alignment(horizontal="center", vertical="center")
    ws.row_dimensions[1].height = 20

    # ── column headers ─────────────────────────────────────────────────────
    cols = [
        ("Date", 13), ("Day", 5), ("Showtime", 10), ("Film Title", 40),
        ("Film Year", 8), ("Format", 7), ("Series", 30), ("Venue", 22),
        ("Confidence", 11), ("Source URL", 44),
    ]
    for i, (label, width) in enumerate(cols, 1):
        _hdr(ws.cell(row=2, column=i), label)
        ws.column_dimensions[get_column_letter(i)].width = width
    ws.row_dimensions[2].height = 20

    # ── data rows ──────────────────────────────────────────────────────────
    for r_idx, s in enumerate(screenings, 3):
        bg = _CONF_COLORS.get(s.confidence, "DAEEF3")
        row_vals = [
            s.date, s.day, s.showtime, s.film_title, s.film_year,
            s.format_, s.series, s.venue, s.confidence, s.source_url,
        ]
        for c_idx, val in enumerate(row_vals, 1):
            _data(
                ws.cell(row=r_idx, column=c_idx),
                val, bg=bg,
                wrap=(c_idx in (4, 7, 10)),
                center=(c_idx in (2, 5, 6, 9)),
            )
        ws.row_dimensions[r_idx].height = 15

    ws.freeze_panes = "A3"
    ws.auto_filter.ref = f"A2:J{len(screenings) + 2}"
    wb.save(path)
    log.info("Saved %s  (%d rows)", path, len(screenings))


# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────

def main():
    parser = argparse.ArgumentParser(
        description="Scrape Trylon Cinema showtimes and write to Excel."
    )
    parser.add_argument("--out-dir",    default=".",    help="Output directory")
    parser.add_argument("--delay",      type=float, default=1.2,
                        help="Seconds between HTTP requests (default: 1.2)")
    parser.add_argument("--start-year", type=int,   default=2021,
                        help="First year for calendar scrape (default: 2021)")
    parser.add_argument("--end-year",   type=int,   default=2025,
                        help="Last year for calendar scrape (default: 2025)")
    parser.add_argument("--max-perisphere-pages", type=int, default=50,
                        help="Max Perisphere index pages to walk (default: 50)")
    parser.add_argument("--skip-series",     action="store_true")
    parser.add_argument("--skip-calendar",   action="store_true")
    parser.add_argument("--skip-perisphere", action="store_true")
    args = parser.parse_args()

    global DELAY
    DELAY = args.delay

    out = Path(args.out_dir)
    out.mkdir(parents=True, exist_ok=True)

    all_screenings: list[TrylonScreening] = []

    # ── 1. Series pages (best source) ──────────────────────────────────────
    if not args.skip_series:
        log.info("=== SOURCE 1: SERIES PAGES ===")
        series_urls = get_series_urls()
        for url in series_urls:
            all_screenings.extend(scrape_series_page(url))

    # ── 2. Calendar months ─────────────────────────────────────────────────
    if not args.skip_calendar:
        log.info("=== SOURCE 2: CALENDAR (%d–%d) ===", args.start_year, args.end_year)
        all_screenings.extend(
            scrape_all_calendar_months(args.start_year, args.end_year)
        )

    # ── 3. Perisphere blog ─────────────────────────────────────────────────
    if not args.skip_perisphere:
        log.info("=== SOURCE 3: PERISPHERE BLOG ===")
        post_urls = get_perisphere_post_urls(max_pages=args.max_perisphere_pages)
        for url in post_urls:
            all_screenings.extend(scrape_perisphere_post(url))

    # ── Deduplicate, sort, write ────────────────────────────────────────────
    final = dedupe_and_sort(all_screenings)
    log.info("Total unique screenings: %d", len(final))

    out_path = str(out / "trylon_screenings.xlsx")
    write_xlsx(final, out_path)
    log.info("Done.  Output: %s", out_path)


if __name__ == "__main__":
    main()